In [2]:
import sys
from pathlib import Path

# Notebook jest w notebooks/, więc root repo to parent
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import ee
import geemap
import geopandas as gpd

from src.config import UserParams, Paths
from src.pipeline import bootstrap, get_area_geometry

# Parametry (źródło prawdy)
params = UserParams()

# Notebook ma zapisywać lokalnie do notebooks/outputs/
paths = Paths(outputs_dir=Path("outputs"))

# Inicjalizacja GEE + przygotowanie folderów
bootstrap(params, paths)

# Geometria obszaru (raz)
area = get_area_geometry(params)

In [3]:
# VISUALIZATION

m = geemap.Map()
m.add_basemap("Esri.WorldImagery")

area_outline = ee.FeatureCollection(area).style(
    color="red",
    width=2,
    fillColor="00000000"
)

m.add_layer(area_outline, {}, "Granica obszaru (AOI)")
m.centerObject(area, 12)
m

Map(center=[50.45758765104263, 17.353593719899337], controls=(WidgetControl(options=['position', 'transparent_…

In [4]:
# TIME WINDOW

event_date_str = params.event_date_str
days_before = params.days_before
days_after = params.days_after

print("Event date:", event_date_str)
print("Window:", days_before, "days before,", days_after, "days after")

Event date: 2024-09-13
Window: 12 days before, 6 days after


In [5]:
# FLOOD DETECTION

from src.gee_processing import detect_flood_from_s1

flood = detect_flood_from_s1(
    area=area,
    event_date_str=params.event_date_str,
    days_before=params.days_before,
    days_after=params.days_after,
    flood_ratio_threshold=1.35,
    max_slope=5,
    min_area_m2=800,
)

diffD = flood.diffD
diffA = flood.diffA
floodedD = flood.floodedD_raw
floodedA = flood.floodedA_raw

flood_vectors_filteredD = flood.flood_vectors_filteredD
permament_water_bin = flood.permanent_water_bin 

In [6]:
import osmnx as ox
ox.__version__

'1.9.3'

In [6]:
#OSM + FINAL RESULTS

from src.osm_processing import analyze_osm_roads_flood_intersections

osm_res = analyze_osm_roads_flood_intersections(
    area=area,
    flood_vectors=flood_vectors_filteredD,
    network_type="drive_service",
    roads_crs_projected="EPSG:3857",
    buffer_m=5.0,
)

# Zgodność nazw z eksportem
points_gdf = osm_res.intersection_points
buffers_gdf = osm_res.buffers_5m
roads_in_flood = osm_res.roads_in_flood
roads_outside_flood = osm_res.roads_outside_flood

print("Punkty przecięć:", len(points_gdf))
print("Zalane drogi:", osm_res.flooded_length_m / 1000, "km")
print("Niezalane drogi:", osm_res.dry_length_m / 1000, "km")

Punkty przecięć: 5791
Zalane drogi: 109.97370783193497 km
Niezalane drogi: 2042.2724658699808 km


In [7]:
# RESULTS EXPORT

from src.export_layers import export_gdf, export_ee_fc_as_shp, export_permanent_water_shp

export_gdf(points_gdf, paths.outputs_dir, "intersection_points.shp")
export_gdf(buffers_gdf, paths.outputs_dir, "buffers_5m.shp")
export_gdf(roads_in_flood, paths.outputs_dir, "drogi_zalane.shp")
export_gdf(roads_outside_flood, paths.outputs_dir, "drogi_niezalane.shp")

export_ee_fc_as_shp(flood.flood_vectors_filteredD, paths.outputs_dir, "zalane_sar.shp")
export_permanent_water_shp(flood.permanent_water_bin, area, paths.outputs_dir, "permanent_water.shp")

print("Eksport zakończony. Wyniki w:", paths.outputs_dir.resolve())

Eksport zakończony. Wyniki w: D:\Studia\Inżynierka\SAR-Flood-Detection\notebooks\outputs
